# NLLB-200 — Spanish ↔ English General Base Model
**iDISC Personalized Translation Engines · Phase 1: Baseline**

This notebook sets up Meta's NLLB-200 as the **General Base Model (GBM)** — the standard benchmark all future domain-specific fine-tuned models (Automotive, Legal, Medical) will be measured against.

---
**Model variants** (change `MODEL_NAME` in the config cell):
| Variant | VRAM | Notes |
|---|---|---|
| `distilled-600M` | ~3 GB | Fast, good quality |
| `distilled-1.3B` | ~5 GB | Recommended |
| `1.3B` | ~6 GB | Slightly better than distilled |
| `3.3B` | ~14 GB | Best quality |

> Enable GPU accelerator in *Settings → Accelerator → GPU T4 x2* before running.

In [1]:
%%capture
!pip install transformers sentencepiece sacrebleu accelerate bitsandbytes

### Configuration

In [2]:
import logging
import time
from dataclasses import dataclass, field
from typing import Optional

import torch
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer

# ── Config ────────────────────────────────────────────────────────────────────
# Using full 3.3B model since Tesla T4 has 15.6 GB VRAM
MODEL_NAME = "facebook/nllb-200-3.3B"   # Best quality
MAX_NEW_TOKENS = 512
LOAD_IN_8BIT = False   # Not needed

LANG_CODES = {
    "es": "spa_Latn",
    "en": "eng_Latn",
}

logging.basicConfig(level=logging.INFO, format="%(levelname)s | %(message)s")
log = logging.getLogger(__name__)

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {device}")
if device == "cuda":
    print(f"GPU   : {torch.cuda.get_device_name(0)}")
    print(f"VRAM  : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

Device: cuda
GPU   : Tesla T4
VRAM  : 15.6 GB


## Data Classes

In [3]:
@dataclass
class TranslationResult:
    source_text: str
    translated_text: str
    source_lang: str
    target_lang: str
    latency_ms: float
    device: str
    model: str = MODEL_NAME
    # Fields populated later by the QA module (Phase 2)
    score: Optional[float] = None
    flagged_for_review: bool = False


@dataclass
class BatchTranslationResult:
    results: list = field(default_factory=list)
    total_latency_ms: float = 0.0
    avg_latency_ms: float = 0.0

In [4]:
class NLLBTranslator:
    """
    General Base Model wrapper for Meta NLLB-200 (Spanish ↔ English).
    Serves as the standard benchmark for domain-specific fine-tuning.
    """

    def __init__(self, model_name: str = MODEL_NAME, load_in_8bit: bool = LOAD_IN_8BIT):
        self.model_name = model_name
        self.device = "cuda" if torch.cuda.is_available() else "cpu"

        log.info(f"Loading tokenizer: {model_name}")
        self.tokenizer = AutoTokenizer.from_pretrained(model_name)

        log.info(f"Loading model → {self.device}")
        load_kwargs = {"torch_dtype": torch.float16} if self.device == "cuda" else {}
        if load_in_8bit and self.device == "cuda":
            load_kwargs["load_in_8bit"] = True

        self.model = AutoModelForSeq2SeqLM.from_pretrained(model_name, **load_kwargs)
        if not load_in_8bit:
            self.model = self.model.to(self.device)
        self.model.eval()
        if self.device == "cuda":
            self.model = torch.compile(self.model)
        log.info("Model ready")

    # ── Single segment ────────────────────────────────────────────────────────

    def translate(
            self,
        text: str,
        src: str = "es",
        tgt: str = "en",
        max_new_tokens: int = 128,          # reduced from 512
        num_beams: int = 4,                 # reduced from 5
        no_repeat_ngram_size: int = 4,      # increased from 3
        repetition_penalty: float = 1.3,    # added
    ) -> TranslationResult:
        if src not in LANG_CODES or tgt not in LANG_CODES:
            raise ValueError(f"Unsupported pair: {src}→{tgt}. Use 'es' or 'en'.")
    
        # --- Tag masking ---
        masked_text, tag_map = mask_tags(text)
    
        self.tokenizer.src_lang = LANG_CODES[src]
        forced_bos_id = self.tokenizer.convert_tokens_to_ids(LANG_CODES[tgt])
    
        inputs = self.tokenizer(
            masked_text, return_tensors="pt", padding=True,
            truncation=True, max_length=1024
        ).to(self.device)
    
        t0 = time.perf_counter()
        with torch.no_grad():
            output_ids = self.model.generate(
                **inputs,
                forced_bos_token_id=forced_bos_id,
                max_new_tokens=max_new_tokens,
                num_beams=num_beams,
                no_repeat_ngram_size=no_repeat_ngram_size,
                repetition_penalty=repetition_penalty,
                early_stopping=True,
            )
        latency_ms = (time.perf_counter() - t0) * 1000
    
        translated_masked = self.tokenizer.decode(output_ids[0], skip_special_tokens=True)
        # --- Tag restoration ---
        translated_text = restore_tags(translated_masked, tag_map)
    
        return TranslationResult(
            source_text=text,
            translated_text=translated_text,
            source_lang=src,
            target_lang=tgt,
            latency_ms=round(latency_ms, 2),
            device=self.device,
        )


    # ── Batch ─────────────────────────────────────────────────────────────────

    def translate_batch(
        self,
        texts: list,
        src: str = "es",
        tgt: str = "en",
        batch_size: int = 8,
        **kwargs,
    ) -> BatchTranslationResult:
        """Translate a list of segments with GPU batching."""
        self.tokenizer.src_lang = LANG_CODES[src]
        forced_bos_id = self.tokenizer.convert_tokens_to_ids(LANG_CODES[tgt])
        all_results, total_start = [], time.perf_counter()

        for i in range(0, len(texts), batch_size):
            batch = texts[i : i + batch_size]
            inputs = self.tokenizer(
                batch, return_tensors="pt", padding=True,
                truncation=True, max_length=1024
            ).to(self.device)

            t0 = time.perf_counter()
            with torch.no_grad():
                output_ids = self.model.generate(
                    **inputs,
                    forced_bos_token_id=forced_bos_id,
                    max_new_tokens=kwargs.get("max_new_tokens", MAX_NEW_TOKENS),
                    num_beams=kwargs.get("num_beams", 5),
                    no_repeat_ngram_size=kwargs.get("no_repeat_ngram_size", 3),
                    early_stopping=True,
                )
            batch_latency = (time.perf_counter() - t0) * 1000
            per_seg = batch_latency / len(batch)

            for src_text, tgt_text in zip(
                batch, self.tokenizer.batch_decode(output_ids, skip_special_tokens=True)
            ):
                all_results.append(TranslationResult(
                    source_text=src_text, translated_text=tgt_text,
                    source_lang=src, target_lang=tgt,
                    latency_ms=round(per_seg, 2), device=self.device,
                ))
            log.info(f"Batch {i // batch_size + 1}: {len(batch)} segs in {batch_latency:.0f} ms")

        total_ms = (time.perf_counter() - total_start) * 1000
        return BatchTranslationResult(
            results=all_results,
            total_latency_ms=round(total_ms, 2),
            avg_latency_ms=round(total_ms / len(texts), 2) if texts else 0.0,
        )

    # ── GPU info ──────────────────────────────────────────────────────────────

    def gpu_info(self) -> dict:
        if not torch.cuda.is_available():
            return {"device": "cpu"}
        return {
            "device":       torch.cuda.get_device_name(0),
            "allocated_gb": round(torch.cuda.memory_allocated(0) / 1e9, 2),
            "reserved_gb":  round(torch.cuda.memory_reserved(0)  / 1e9, 2),
            "total_gb":     round(torch.cuda.get_device_properties(0).total_memory / 1e9, 2),
        }

In [5]:
import re
from typing import Tuple, Dict

def mask_tags(text: str) -> Tuple[str, Dict[str, str]]:
    """Replace XML/HTML tags with unique placeholders."""
    tag_map = {}
    def repl(match):
        tag = match.group(0)
        idx = len(tag_map)
        placeholder = f"__TAG_{idx}__"
        tag_map[placeholder] = tag
        return placeholder
    masked = re.sub(r'<[^>]+>', repl, text)
    return masked, tag_map

def restore_tags(text: str, tag_map: Dict[str, str]) -> str:
    """Restore placeholders back to original tags."""
    for placeholder, tag in tag_map.items():
        text = text.replace(placeholder, tag)
    return text

## Load Model

In [6]:
translator = NLLBTranslator()

INFO | Loading tokenizer: facebook/nllb-200-3.3B
INFO | HTTP Request: HEAD https://huggingface.co/facebook/nllb-200-3.3B/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"
INFO | HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/facebook/nllb-200-3.3B/1a07f7d195896b2114afcb79b7b57ab512e7b43e/config.json "HTTP/1.1 200 OK"
INFO | HTTP Request: GET https://huggingface.co/api/resolve-cache/models/facebook/nllb-200-3.3B/1a07f7d195896b2114afcb79b7b57ab512e7b43e/config.json "HTTP/1.1 200 OK"


config.json:   0%|          | 0.00/808 [00:00<?, ?B/s]

INFO | HTTP Request: HEAD https://huggingface.co/facebook/nllb-200-3.3B/resolve/main/tokenizer_config.json "HTTP/1.1 307 Temporary Redirect"
WARNING | Warning: You are sending unauthenticated requests to the HF Hub. Please set a HF_TOKEN to enable higher rate limits and faster downloads.
INFO | HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/facebook/nllb-200-3.3B/1a07f7d195896b2114afcb79b7b57ab512e7b43e/tokenizer_config.json "HTTP/1.1 200 OK"
INFO | HTTP Request: GET https://huggingface.co/api/resolve-cache/models/facebook/nllb-200-3.3B/1a07f7d195896b2114afcb79b7b57ab512e7b43e/tokenizer_config.json "HTTP/1.1 200 OK"


tokenizer_config.json:   0%|          | 0.00/564 [00:00<?, ?B/s]

INFO | HTTP Request: HEAD https://huggingface.co/facebook/nllb-200-3.3B/resolve/main/tokenizer_config.json "HTTP/1.1 307 Temporary Redirect"
INFO | HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/facebook/nllb-200-3.3B/1a07f7d195896b2114afcb79b7b57ab512e7b43e/tokenizer_config.json "HTTP/1.1 200 OK"
INFO | HTTP Request: GET https://huggingface.co/api/models/facebook/nllb-200-3.3B/tree/main/additional_chat_templates?recursive=false&expand=false "HTTP/1.1 404 Not Found"
INFO | HTTP Request: GET https://huggingface.co/api/models/facebook/nllb-200-3.3B/tree/main?recursive=true&expand=false "HTTP/1.1 200 OK"
INFO | HTTP Request: HEAD https://huggingface.co/facebook/nllb-200-3.3B/resolve/main/tokenizer.json "HTTP/1.1 302 Found"
INFO | HTTP Request: GET https://huggingface.co/api/models/facebook/nllb-200-3.3B/xet-read-token/1a07f7d195896b2114afcb79b7b57ab512e7b43e "HTTP/1.1 200 OK"


...

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Loading weights:   0%|          | 0/1016 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie model.shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie model.shared.weight to model.decoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie model.shared.weight to model.encoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
INFO | HTTP Request: HEAD https://huggingface.co/facebook/nllb-200-3.3B/resolve/main/generation_config.json "HTTP/1.1 307 Temporary Redirect"
INFO | HTTP Request: HEAD https://huggingfa

generation_config.json:   0%|          | 0.00/189 [00:00<?, ?B/s]

INFO | Model ready


In [7]:
# Spanish → English
text_es = "El paciente tiene fiebre y dolor de cabeza."
result_en = translator.translate(text_es, src="es", tgt="en")
print(f"Spanish : {text_es}")
print(f"English : {result_en.translated_text}")
print(f"Latency : {result_en.latency_ms} ms\n")

# English → Spanish
text_en = "The engine requires an oil change every 5,000 miles."
result_es = translator.translate(text_en, src="en", tgt="es")
print(f"English : {text_en}")
print(f"Spanish : {result_es.translated_text}")
print(f"Latency : {result_es.latency_ms} ms")

Spanish : El paciente tiene fiebre y dolor de cabeza.
English : The patient has fever and headache. The patient has a fever and a headache".
Latency : 2804.09 ms

English : The engine requires an oil change every 5,000 miles.
Spanish : El motor requiere un cambio de aceite cada 5,000 millas.
Latency : 588.84 ms


### Baseline Benchmark

In [8]:
import pandas as pd
import sacrebleu

# Benchmark Cases with Human References and Tag Test
# (text, src, tgt, domain, reference)
benchmark_cases = [
    ("El cielo es azul y el sol brilla.", "es", "en", "General", "The sky is blue and the sun shines."),    
    ("The quick brown fox jumps over the lazy dog.", "en", "es", "General", "El rápido zorro marrón salta sobre el perro perezoso."),    
    ("El paciente presenta fiebre alta y tos seca.", "es", "en", "Medical", "The patient presents with high fever and a dry cough."),    
    # TAG INTEGRITY TEST: Essential for iDISC's requirement
    ("El paciente tiene <bold>fiebre</bold>.", "es", "en", "Technical", "The patient has <bold>fever</bold>."),    
    ("The torque converter must be replaced.", "en", "es", "Automotive", "El convertidor de par debe ser reemplazado."),
]

# Execution and Metric Calculation
results_for_export = []

print(f"{'Domain':<12} {'Dir':<8} {'BLEU':>7} {'Latency':>10}  Source / Translation")
print("─" * 90)

for text, src, tgt, domain, reference in benchmark_cases:
    # Perform translation
    r = translator.translate(text, src=src, tgt=tgt)
    
    # Calculate Sentence-level BLEU (for baseline report)
    # sacrebleu expects lists for metrics
    bleu = sacrebleu.sentence_bleu(r.translated_text, [reference]).score
    
    direction = f"{src.upper()}→{tgt.upper()}"
    print(f"[{domain:<10}] {direction:<8} {bleu:>7.2f} {r.latency_ms:>8.0f} ms")
    print(f"  SRC: {r.source_text}")
    print(f"  TGT: {r.translated_text}")
    print(f"  REF: {reference}\n")

    # Collect data for CSV
    results_for_export.append({
        "Domain": domain,
        "Direction": direction,
        "Source": r.source_text,
        "Translation": r.translated_text,
        "Reference": reference,
        "BLEU_Score": round(bleu, 2),
        "Latency_ms": r.latency_ms,
        "Model": MODEL_NAME
    })

# Export to CSV (The Baseline Report)
df_baseline = pd.DataFrame(results_for_export)
csv_filename = "Baseline_Report.csv"
df_baseline.to_csv(csv_filename, index=False, encoding='utf-8-sig')

print(f"Baseline report exported to: {csv_filename}")

Domain       Dir         BLEU    Latency  Source / Translation
──────────────────────────────────────────────────────────────────────────────────────────
[General   ] ES→EN      66.06      503 ms
  SRC: El cielo es azul y el sol brilla.
  TGT: The sky is blue and the sun is shining.
  REF: The sky is blue and the sun shines.

[General   ] EN→ES      11.79      998 ms
  SRC: The quick brown fox jumps over the lazy dog.
  TGT: El rapido zorro marrón y veloz, el rápido zorro pardo, salta por encima del perro perezoso.
  REF: El rápido zorro marrón salta sobre el perro perezoso.

[Medical   ] ES→EN      70.17      583 ms
  SRC: El paciente presenta fiebre alta y tos seca.
  TGT: The patient presents a high fever and a dry cough.
  REF: The patient presents with high fever and a dry cough.

[Technical ] ES→EN       0.68     3696 ms
  SRC: El paciente tiene <bold>fiebre</bold>.
  TGT: The patient has been diagnosed with this disease, and the patient has had this disease. . . . . ... ... ... 

#### !! no tag preservation right now

### Batch Translation

In [9]:
segments = [
    "El contrato incluye una cláusula de confidencialidad.",
    "El vehículo debe pasar una inspección técnica anual.",
    "Se recomienda reposo absoluto durante 48 horas.",
    "La sentencia fue apelada ante el tribunal superior.",
]

batch_result = translator.translate_batch(segments, src="es", tgt="en", batch_size=4)

print(f"Total  : {batch_result.total_latency_ms:.0f} ms")
print(f"Avg    : {batch_result.avg_latency_ms:.0f} ms / segment\n")

for r in batch_result.results:
    print(f"  ES: {r.source_text}")
    print(f"  EN: {r.translated_text}\n")

INFO | Batch 1: 4 segs in 771 ms


Total  : 774 ms
Avg    : 194 ms / segment

  ES: El contrato incluye una cláusula de confidencialidad.
  EN: The contractual contract includes a confidentiality clause in the contract.

  ES: El vehículo debe pasar una inspección técnica anual.
  EN: The vehicle must pass an annual technical technical inspection every year.

  ES: Se recomienda reposo absoluto durante 48 horas.
  EN: Absolute rest is recommended for 48 hours with 48 hours of 48 hours rest.

  ES: La sentencia fue apelada ante el tribunal superior.
  EN: The sentence was appealed against against against the sentence before the higher court.



---
## Next steps
1. General Base Model (this notebook)
2. QA Module — COMET + BERTScore thresholds
3. Fine-tuning pipelines (Medical / Legal / Automotive)
4. Automated Repair — Mistral-7B + T5 tag repair
5. Streamlit / Gradio prototype with heatmap UI